# Reusable template — binary logistic regression on an energy blotter

**Short name:** `Energy_LogReg`  
Swap path, positive label, and features. Keep: *clean → imbalance → dummies → scale check → L1 fit → threshold → ROC → simulate*.

Teaching use only. Not an operating reliability tool.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ---- edit these ----
DATA_PATH = "data/energy_units.csv"
TARGET = "forced_outage"
POSITIVE = "Yes"
FEATURES = ["age_years", "ambient_c", "capacity_mw", "load_factor",
            "days_since_maint", "is_peak_hour", "fuel", "region"]
TEST_SIZE = 0.20
SEED = 1
C = 0.05
THRESHOLD = 0.50
SCALE = False
# --------------------

df = pd.read_csv(DATA_PATH)
y = (df[TARGET] == POSITIVE).astype(int).to_numpy()
X = pd.get_dummies(df[list(dict.fromkeys(FEATURES))], drop_first=True).astype(float)
print(df[TARGET].value_counts(normalize=True))
print("X", X.shape, "event rate", y.mean())

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
if SCALE:
    model = Pipeline([
        ("sc", StandardScaler()),
        ("lr", LogisticRegression(C=C, penalty="l1", solver="liblinear")),
    ])
else:
    model = LogisticRegression(C=C, penalty="l1", solver="liblinear")
model.fit(Xtr, ytr)
lr = model.named_steps["lr"] if SCALE else model
proba = model.predict_proba(Xte)[:, 1]
pred = (proba >= THRESHOLD).astype(int)
print("intercept", float(lr.intercept_[0]))
print("acc", accuracy_score(yte, pred),
      "prec", precision_score(yte, pred, zero_division=0),
      "rec", recall_score(yte, pred, zero_division=0),
      "auc", roc_auc_score(yte, proba))
print(confusion_matrix(yte, pred))
print(pd.DataFrame({"var": Xtr.columns, "coef": lr.coef_[0]})
      .query("coef.abs() > 0").sort_values("coef"))
fpr, tpr, _ = roc_curve(yte, proba)
plt.figure(figsize=(5, 5)); plt.plot(fpr, tpr); plt.plot([0, 1], [0, 1], "--")
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC")
plt.show()
